In [1]:
import requests
import time
import random
import hashlib

# 20 stocks
symbols = [
    "RELIANCE","TCS","INFY","HDFCBANK","ICICIBANK",
    "SBIN","LT","ITC","HINDUNILVR","KOTAKBANK",
    "AXISBANK","ASIANPAINT","MARUTI","SUNPHARMA",
    "TITAN","ULTRACEMCO","BAJFINANCE","WIPRO",
    "NTPC","POWERGRID"
]

# Misra-Gries
k = 5
mg_counter = {}

# FM
max_zeros = 0

# HLL
b = 4
m = 2 ** b
registers = [0] * m

In [2]:
def fetch_nse(symbol):
    url = f"https://www.nseindia.com/api/quote-equity?symbol={symbol}"

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Referer": "https://www.nseindia.com/",
        "Connection": "keep-alive"
    }

    try:
        session = requests.Session()
        session.get("https://www.nseindia.com", headers=headers)  # important

        response = session.get(url, headers=headers)

        if response.status_code != 200:
            return None

        return response.json()

    except:
        return None

In [3]:
def update_mg(item):
    global mg_counter

    if item in mg_counter:
        mg_counter[item] += 1
    elif len(mg_counter) < k:
        mg_counter[item] = 1
    else:
        keys = list(mg_counter.keys())
        for key in keys:
            mg_counter[key] -= 1
            if mg_counter[key] == 0:
                del mg_counter[key]

In [4]:
def hash_value(item):
    return int(hashlib.md5(item.encode()).hexdigest(), 16)

def trailing_zeros(x):
    if x == 0:
        return 0
    return (x & -x).bit_length() - 1

def update_fm(item):
    global max_zeros
    h = hash_value(item)
    tz = trailing_zeros(h)
    max_zeros = max(max_zeros, tz)

def estimate_distinct():
    return 2 ** max_zeros

In [5]:
def update_hll(item):
    global registers

    h = hash_value(item)

    bucket = h & (m - 1)
    w = h >> b

    tz = trailing_zeros(w)

    registers[bucket] = max(registers[bucket], tz)

def estimate_hll():
    Z = sum([2 ** (-r) for r in registers])
    E = (0.7213 / (1 + 1.079 / m)) * (m ** 2) / Z
    return int(E)